# HDBSCAN Diagnostic — ONE author
Time each step to find the bottleneck.

In [ ]:
import time, numpy as np
from sklearn.cluster import HDBSCAN
from sklearn.metrics.pairwise import cosine_distances

AUTHOR_ID = 5074251636  # dental + soil overmerge, 69 works
results = []

# Step 1: Get work_ids from work_authors
t0 = time.time()
work_ids = [r.work_id for r in spark.sql(f"""
    SELECT work_id FROM openalex.works.work_authors WHERE author_id = {AUTHOR_ID}
""").collect()]
t1 = time.time()
results.append(f'Step 1 (get work_ids): {t1-t0:.1f}s — {len(work_ids)} works')
print(results[-1])

# Step 2: Get embeddings by work_id list
ids_str = ','.join(f"'{w}'" for w in work_ids)
t2 = time.time()
rows = spark.sql(f"""
    SELECT work_id, embedding
    FROM openalex.vector_search.work_embeddings_v2
    WHERE work_id IN ({ids_str})
      AND embedding IS NOT NULL
""").collect()
t3 = time.time()
results.append(f'Step 2 (get embeddings): {t3-t2:.1f}s — {len(rows)} embeddings')
print(results[-1])

# Step 3: Convert to numpy
t4 = time.time()
X = np.array([np.array(r.embedding[:1024], dtype=np.float32) for r in rows])
t5 = time.time()
results.append(f'Step 3 (to numpy): {t5-t4:.1f}s — shape {X.shape}')
print(results[-1])

# Step 4: Run HDBSCAN (precomputed cosine distance — sklearn doesn't support metric='cosine')
t6 = time.time()
dist_matrix = cosine_distances(X)
clusterer = HDBSCAN(min_cluster_size=3, min_samples=2, metric='precomputed')
labels = clusterer.fit_predict(dist_matrix)
t7 = time.time()
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
results.append(f'Step 4 (HDBSCAN): {t7-t6:.1f}s — {n_clusters} clusters, labels={list(set(labels))}')
print(results[-1])

# Step 5: Now try the JOIN approach (what the real notebook does)
t8 = time.time()
joined = spark.sql(f"""
    SELECT t.author_id, e.work_id, e.embedding
    FROM openalex.aer.temp_hdbscan_work_ids t
    JOIN openalex.vector_search.work_embeddings_v2 e ON t.work_id = e.work_id
    WHERE t.author_id = {AUTHOR_ID}
      AND e.embedding IS NOT NULL
""").collect()
t9 = time.time()
results.append(f'Step 5 (staged table JOIN): {t9-t8:.1f}s — {len(joined)} rows')
print(results[-1])

# Step 6: Try the JOIN for 20 authors (return just dims, not full embeddings)
t10 = time.time()
batch20 = spark.sql(f"""
    SELECT t.author_id, e.work_id, size(e.embedding) AS dims
    FROM openalex.aer.temp_hdbscan_work_ids t
    JOIN openalex.vector_search.work_embeddings_v2 e ON t.work_id = e.work_id
    WHERE t.author_id IN (5000058354,5000075346,5000163809,5000185973,5000277470,
        5000294851,5000296333,5000304681,5000447750,5000469744,
        5000470139,5000484636,5000497992,5000843678,5000848656,
        5000904925,5000911921,5001003834,5001032099,5001106162)
      AND e.embedding IS NOT NULL
""").collect()
t11 = time.time()
results.append(f'Step 6 (20-author JOIN, dims only): {t11-t10:.1f}s — {len(batch20)} rows')
print(results[-1])

# Step 7: Same 20 authors but WITH full embeddings
t12 = time.time()
batch20_full = spark.sql(f"""
    SELECT t.author_id, e.work_id, e.embedding
    FROM openalex.aer.temp_hdbscan_work_ids t
    JOIN openalex.vector_search.work_embeddings_v2 e ON t.work_id = e.work_id
    WHERE t.author_id IN (5000058354,5000075346,5000163809,5000185973,5000277470,
        5000294851,5000296333,5000304681,5000447750,5000469744,
        5000470139,5000484636,5000497992,5000843678,5000848656,
        5000904925,5000911921,5001003834,5001032099,5001106162)
      AND e.embedding IS NOT NULL
""").collect()
t13 = time.time()
results.append(f'Step 7 (20-author JOIN, full embeddings): {t13-t12:.1f}s — {len(batch20_full)} rows')
print(results[-1])

output = '\n'.join(results)
print('\n=== SUMMARY ===')
print(output)
dbutils.notebook.exit(output)